# 14 - Does scaling BEFORE iterative imputation help the SVM?

Our anchor SVM (LB 0.85456) imputes first, then scales:
`IterativeImputer(BayesianRidge) -> StandardScaler -> SVC`.

The imputer's inner model is a **regularized** linear regression (`BayesianRidge`), and
its priors are scale-sensitive. Our features are pre-standardized but their spreads still
range **std ~1-5**, so during imputation the regularization hits each column unequally.
Scaling *before* the imputer equalizes that. This notebook tests whether it matters,
using the project's standard bar: repeated CV + paired t-test, adopt only if the gain
clears ~+/-0.008 fold noise.

| Pipeline | imputer sees | order |
|---|---|---|
| **A (current)** | raw features | impute -> scale -> SVM |
| **B (pre-scale)** | standardized features | scale -> impute -> scale -> SVM |

Both end with the *same* `StandardScaler -> SVC`, so the only thing that differs is the
scale of the data the imputer regresses on - a clean isolation of the effect.
`StandardScaler` handles NaN natively (ignored in fit, preserved in transform), so
putting it before the imputer is valid.

## Step 0 - Imports and data

In [1]:
import warnings; warnings.filterwarnings('ignore')
import os, numpy as np, pandas as pd
from sklearn.experimental import enable_iterative_imputer   # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import f1_score
from scipy import stats

TRAIN_PATH, TEST_PATH, OUT_DIR = '../train.csv', '../test.csv', '../outputs'
train = pd.read_csv(TRAIN_PATH); test = pd.read_csv(TEST_PATH)
FEATURES = [c for c in train.columns if c not in ('id','sleep_stage')]
X, y, Xtest = train[FEATURES], train['sleep_stage'].to_numpy(), test[FEATURES]
print('train', train.shape, '| test', test.shape)
print('feature std range: %.2f - %.2f' % (X.std().min(), X.std().max()))

train (9000, 23) | test (5000, 22)
feature std range: 1.04 - 5.04


## Step 1 - The two pipelines
`probability=False`: we only need predicted labels for macro-F1, so we skip the slow
internal calibration - this makes the repeated CV much faster.

In [2]:
SVM_PARAMS = dict(kernel='rbf', C=12, gamma=0.012, random_state=42)
def imputer(): return IterativeImputer(estimator=BayesianRidge(), max_iter=10, random_state=42)

def pipe_A():   # current anchor: impute on raw -> scale -> SVM
    return make_pipeline(imputer(), StandardScaler(), SVC(**SVM_PARAMS))

def pipe_B():   # pre-scale: scale -> impute on standardized -> scale -> SVM
    return make_pipeline(StandardScaler(), imputer(), StandardScaler(), SVC(**SVM_PARAMS))

## Step 2 - Repeated CV + paired t-test (the decision)
For each fold seed we get OOF predictions for A and B over the *same* folds, score each
**per fold**, and run a **paired** t-test. Both pipelines refit their imputer/scaler
inside every fold, so there is no leakage. (A few minutes; set `SEEDS=[42]` to preview.)

In [3]:
SEEDS = [42, 2024, 7]
A_fold, B_fold = [], []
for seed in SEEDS:
    cv = StratifiedKFold(5, shuffle=True, random_state=seed)
    oof_A = cross_val_predict(pipe_A(), X, y, cv=cv, method='predict', n_jobs=5)
    oof_B = cross_val_predict(pipe_B(), X, y, cv=cv, method='predict', n_jobs=5)
    for _, idx in cv.split(X, y):
        A_fold.append(f1_score(y[idx], oof_A[idx], average='macro'))
        B_fold.append(f1_score(y[idx], oof_B[idx], average='macro'))
    print('seed %-4d  A(scale-after)=%.4f  B(scale-before)=%.4f'
          % (seed, f1_score(y, oof_A, average='macro'),
                   f1_score(y, oof_B, average='macro')), flush=True)

A_fold, B_fold = np.array(A_fold), np.array(B_fold)
gain = B_fold - A_fold
print('\nmean gain (B - A) = %+.4f +/- %.4f over %d folds' % (gain.mean(), gain.std(), len(gain)))
print('B wins %d/%d folds   paired t-test p = %.2e'
      % ((gain > 0).sum(), len(gain), stats.ttest_rel(B_fold, A_fold).pvalue))
print('\n-> Adopt B only if the gain is clearly > ~0.008 and it wins most folds.')

seed 42    A(scale-after)=0.8407  B(scale-before)=0.8407
seed 2024  A(scale-after)=0.8402  B(scale-before)=0.8401
seed 7     A(scale-after)=0.8432  B(scale-before)=0.8431

mean gain (B - A) = -0.0001 +/- 0.0003 over 15 folds
B wins 1/15 folds   paired t-test p = 2.77e-01

-> Adopt B only if the gain is clearly > ~0.008 and it wins most folds.


## Step 3 - Submission for pipeline B (use only if Step 2 cleared the bar)
Writes `svm_prescale_submission.csv`. If Step 2 was flat/negative, this just confirms
the current anchor is already at its scaling optimum - keep `svm_iterimpute` instead.

In [4]:
os.makedirs(OUT_DIR, exist_ok=True)
mdl = pipe_B().fit(X, y)
pred = np.asarray(mdl.predict(Xtest)).astype(int).ravel()
sub = pd.DataFrame({'id': test['id'], 'sleep_stage': pred})
path = os.path.join(OUT_DIR, 'svm_prescale_submission.csv'); sub.to_csv(path, index=False)
print('wrote', path, sub.shape)
print('predicted class distribution:', dict(sub.sleep_stage.value_counts().sort_index()))
sub.head()

wrote ../outputs/svm_prescale_submission.csv (5000, 2)
predicted class distribution: {0: np.int64(1120), 1: np.int64(1290), 2: np.int64(1292), 3: np.int64(1298)}


,id,sleep_stage
0,9000,3
1,9001,3
2,9002,1
3,9003,3
4,9004,3


## Takeaways
- This isolates one variable - the **scale the imputer's BayesianRidge regresses on** -
  with everything downstream held identical, judged by the project's repeated-CV bar.
- Likely a small effect (features are already roughly standardized), but it's a free,
  properly-untried tweak to the anchor model: adopt B if it clears +/-0.008, else you've
  confirmed scale-after-impute was already optimal and ruled it out cleanly.